In [58]:
import geopandas as gpd
import pandas as pd
import re
import os
from rapidfuzz import fuzz, process
from shapely import Point

#### Paths

In [38]:
names_dir = '../../../data/shapes'
farmers_xls = f'{names_dir}/farmers.xlsx'
shp_path = f'{names_dir}/awg_farmers_final.shp'
assert os.path.exists(farmers_xls), 'Check path'

#### Read in excel file and clean the headers

In [39]:
df = pd.read_excel(farmers_xls, skiprows=2, sheet_name='Sheet1')

In [40]:
df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Names of Famers,Trees given,Estimanted Number of\ntrees per acre,Spacing,Species,Acreage,Coordinates
0,NaN,NaN,NaN,1,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E"
1,NaN,NaN,NaN,2,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E"
2,NaN,NaN,NaN,3,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E"
3,NaN,NaN,NaN,4,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E"
4,NaN,NaN,NaN,5,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E"


In [41]:
df = df.iloc[:,4:]

In [42]:
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

In [43]:
df.columns

Index(['names_of_famers', 'trees_given',
       'estimanted_number_of\ntrees_per_acre', 'spacing', 'species', 'acreage',
       'coordinates'],
      dtype='str')

In [44]:
df.rename(columns={'estimanted_number_of\ntrees_per_acre': 'trees_per_acre', 'names_of_famers' : 'name'}, inplace=True )

#### Convert the coordinates from deg min secs to decimal degrees

In [45]:
def parse_dms(dms_str):
    if pd.isna(dms_str): return None, None
    
    # Matches: degrees° minutes' seconds" direction
    pattern = r"(\d+(?:\.\d+)?)°(\d+(?:\.\d+)?)\'(\d+(?:\.\d+)?)\"([NSEW])"
    parts = re.findall(pattern, str(dms_str))
    
    def to_dd(deg, mn, sec, direction):
        dd = float(deg) + float(mn)/60 + float(sec)/3600
        if direction in ['S', 'W']:
            dd *= -1
        return dd

    # Returns (Latitude, Longitude) as floats
    if len(parts) == 2:
        lat = to_dd(*parts[0])
        lon = to_dd(*parts[1])
        return lat, lon
    return None, None

# Usage with your Excel file
df[['lat', 'lon']] = df['coordinates'].apply(lambda x: pd.Series(parse_dms(x)))

In [46]:
df.head()

,name,trees_given,trees_per_acre,spacing,species,acreage,coordinates,lat,lon
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028


In [47]:
df.dropna(subset=['lat', 'lon'], inplace=True)

In [56]:
df['estimated_trees_present'] = round(df['trees_per_acre'] * df['acreage'], ndigits=1)

In [57]:
df['estimated_trees_present'].describe()

count     163.000000
mean      192.797546
std       156.934175
min       100.000000
25%       150.000000
50%       150.000000
75%       150.000000
max      1000.000000
Name: estimated_trees_present, dtype: float64

#### Create geometry column using shapely

In [49]:
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]

In [50]:
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

In [51]:
gdf.head()

,name,trees_given,trees_per_acre,spacing,species,acreage,coordinates,lat,lon,geometry
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167,POINT (37.86517 -1.58233)
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694,POINT (37.88069 -1.5395)
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639,POINT (37.86064 -1.61897)
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306,POINT (37.87131 -1.62325)
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028,POINT (37.86903 -1.61944)


In [52]:
gdf.isna().sum()

name              0
trees_given       0
trees_per_acre    0
spacing           0
species           0
acreage           0
coordinates       0
lat               0
lon               0
geometry          0
dtype: int64

In [53]:
len(gdf)

163

In [54]:
# gdf.to_file(f'{names_dir}/clean_file.fgb')

### Merge with the shapefile

In [61]:
final_gdf = gpd.read_file(shp_path)

In [62]:
final_gdf.head()

,kamiti_cbo,awg_kml,trees_give,trees_aliv,melia_only,bounds_ok,geometry
0,Naomi Nicodemus Mutuku,Naomie Nicodemus Mutuku,121.0,152.0,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,..."
1,Emma Mutie,Emma Mutie,150.0,145.0,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249..."
2,Dorrine Kasyoka Mutua,Dorrine kasyoka Mutua,108.0,115.0,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,..."
3,Kinyao Musembi,Kinyao Musembi,150.0,33.0,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779..."
4,Simon Mumo Mulu,Simon mumo Mulu,150.0,37.0,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017..."


In [63]:
final_gdf.drop(columns=['kamiti_cbo'], inplace=True)

In [64]:
final_gdf.rename(columns={'awg_kml' : 'name'}, inplace=True)

In [65]:
len(final_gdf)

166

#### Name normalization

In [59]:
# Define unwanted substrings / tokens
UNWANTED_TERMS = [
    "awg",
    "farmer",
    "old farmer",
    "new",
    "ond",
    '-'
]

In [60]:
def normalize_name(name):
    if pd.isnull(name):
        return None
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)  # remove punctuation
    for term in sorted(UNWANTED_TERMS, key=len, reverse=True):
        pattern = r'\b' + re.escape(term) + r'\b'
        name = re.sub(pattern, '', name)
    
    # Remove standalone 4-digit years
    name = re.sub(r'\b\d{4}\b', '', name)
    name = re.sub(r'\s+', ' ', name)     # collapse spaces
    return name.strip()

In [66]:
# Remove null names
df = df[df["name"].notnull()]

# Normalize
df["name_clean"] = df["name"].apply(normalize_name)

# Remove empty
df = df[df["name_clean"].notnull()]

In [67]:
# Remove null names
final_gdf = final_gdf[final_gdf["name"].notnull()]

# Split grouped names
final_gdf["name_split"] = final_gdf["name"].str.split(",")

# Explode into individual rows
final_gdf = final_gdf.explode("name_split")

# Normalize
final_gdf["name_clean"] = final_gdf["name_split"].apply(normalize_name)

# Remove empty
final_gdf = final_gdf[final_gdf["name_clean"].notnull()]

In [68]:
len(df)

163

In [69]:
len(final_gdf)

167

In [71]:
final_gdf.loc[final_gdf['geometry'].duplicated()]

,name,trees_give,trees_aliv,melia_only,bounds_ok,geometry,name_split,name_clean
14,Maurice mulaa Manzolo,250.0,250.0,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",Maurice mulaa Manzolo,maurice mulaa manzolo
15,Rose martha Mulaa,350.0,350.0,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",Rose martha Mulaa,rose martha mulaa
16,joel nyamai mulaa,NaN,NaN,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",joel nyamai mulaa,joel nyamai mulaa
23,Wayua Singila,200.0,105.0,T,T,"POLYGON ((37.85664 -1.58264, 37.85687 -1.58272...",Wayua Singila,wayua singila
55,robert Kasoa Munyilu,150.0,22.0,T,T,"POLYGON ((37.87673 -1.55065, 37.87681 -1.55057...",robert Kasoa Munyilu,robert kasoa munyilu
150,Priska martha Mutia,150.0,156.0,T,T,"POLYGON ((37.87066 -1.62294, 37.87081 -1.62283...",Priska martha Mutia,priska martha mutia
162,"jeremiah ngambi mutisya, ruth mumbua willie",NaN,NaN,T,F,"POLYGON ((37.86815 -1.61886, 37.8681 -1.61794,...",ruth mumbua willie,ruth mumbua willie


In [72]:
analysis_gdf = final_gdf.drop_duplicates(subset=['geometry'], keep='first')

In [74]:
analysis_gdf.to_file(f'{names_dir}/analysis.shp', index=False)